# Sentence Reconstruction

The purpose of this project is to take in input a sequence of words corresponding to a random permutation of a given english sentence, and reconstruct the original sentence.

The otuput can be either produced in a single shot, or through an iterative (autoregressive) loop generating a single token at a time.


CONSTRAINTS:
* No pretrained model can be used.
* The neural network models should have less the 20M parameters.
* No postprocessing should be done (e.g. no beamsearch)
* You cannot use additional training data.


BONUS PARAMETERS:

A bonus of 0-2 points will be attributed to incentivate the adoption of models with a low number of parameters.

# Dataset

The dataset is composed by sentences taken from the generics_kb dataset of hugging face. We restricted the vocabolary to the 10K most frequent words, and only took sentences making use of this vocabulary.

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [1]:
!pip install datasets

Download the dataset

In [2]:
from datasets import load_dataset
from keras.layers import TextVectorization
import tensorflow as tf
import numpy as np

np.random.seed(42)
ds = load_dataset('generics_kb',trust_remote_code=True)['train']

2024-06-10 13:35:40.795677: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-10 13:35:40.795817: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-10 13:35:40.961977: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Generating train split:   0%|          | 0/1020868 [00:00<?, ? examples/s]

Filter row with length greater than 8.


In [3]:
ds = ds.filter(lambda row: len(row["generic_sentence"].split(" ")) > 8 )
corpus = [ '<start> ' + row['generic_sentence'].replace(","," <comma>") + ' <end>' for row in ds ]
corpus = np.array(corpus)

Filter:   0%|          | 0/1020868 [00:00<?, ? examples/s]

Create a tokenizer and Detokenizer

In [4]:
tokenizer=TextVectorization( max_tokens=10000, standardize="lower_and_strip_punctuation", encoding="utf-8",) #con il max prende le piu frequenti. ordina i token del vocab dal piu frequente al meno frequente
tokenizer.adapt(corpus)

class TextDetokenizer:
    def __init__(self, vectorize_layer):
        self.vectorize_layer = vectorize_layer
        vocab = self.vectorize_layer.get_vocabulary()
        self.index_to_word = {index: word for index, word in enumerate(vocab)}

    def __detokenize_tokens(self, tokens):
        def check_token(t):
          if t == 3:
            s="<start>"
          elif t == 2:
            s="<end>"
          elif t == 7:
            s="<comma>"
          else:
            s=self.index_to_word.get(t, '[UNK]')
          return s

        return ' '.join([ check_token(token) for token in tokens if token != 0])

    def __call__(self, batch_tokens):
       return [self.__detokenize_tokens(tokens) for tokens in batch_tokens]


detokenizer = TextDetokenizer( tokenizer )
sentences = tokenizer( corpus ).numpy()


Remove from corpus the sentences where any unknow word appears

In [5]:
mask = np.sum( (sentences==1) , axis=1) >= 1
original_data = np.delete( sentences, mask , axis=0)

In [ ]:
original_data.shape

(241236, 28)

Shuffle the sentences

In [ ]:
# shifted_data = np.roll(original_data, 1, axis=1)

In [152]:
from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.sequence import pad_sequences

class DataGenerator(Sequence):
    def __init__(self, data, training=True, batch_size=32, shuffle=True):

        self.data = data
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.training = training
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        data_batch = np.array([self.data[k] for k in indexes])
        result = np.copy(data_batch)

        #shuffle only the relevant positions for each batch
        for i in range(data_batch.shape[0]):
          np.random.shuffle(data_batch[i,1:data_batch[i].argmin() - 1])
        
        # teacher forcing
        targ_in = result[:, :-1]
        targ_out = result[:, 1:]
        test_data = data_batch[:, :-1]
        seq_len=28
        
        # Pad sequences
        data_batch = pad_sequences(data_batch, maxlen=seq_len, padding='post')
        targ_in = pad_sequences(targ_in, maxlen=seq_len, padding='post')
        targ_out = pad_sequences(targ_out, maxlen=seq_len, padding='post')

        # Generate padding masks
        encoder_padding_mask = data_batch != 0
        decoder_padding_mask = targ_in != 0
        
        print(encoder_padding_mask.shape)
        print(decoder_padding_mask.shape)

        # Generate look-ahead masks (lower triangular matrix)
        look_ahead_mask = 1 - tf.linalg.band_part(tf.ones((self.batch_size, seq_len)), -1, 0)
        look_ahead_mask = tf.cast(look_ahead_mask, dtype=tf.bool)
        
        print(look_ahead_mask.shape)
        # print(look_ahead_mask)

        # Combine padding and look-ahead masks
        # decoder_mask = tf.maximum(decoder_padding_mask, look_ahead_mask)


        if(self.training):
            return (data_batch, encoder_padding_mask, targ_in, decoder_padding_mask, look_ahead_mask), targ_out
        else:
            return (data_batch, encoder_padding_mask, test_data, decoder_padding_mask, look_ahead_mask), targ_out

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.data))
        if self.shuffle:
            np.random.shuffle(self.indexes)

In [ ]:
# train_generator = DataGenerator(original_data[:220000])
# test_generator = DataGenerator(original_data[220000:])
# x, y = test_generator.__getitem__(1)
# x = detokenizer(x[0])
# y = detokenizer(y)

# for i in range(7):
#   print("original: ", y[i])
#   print("shuffled: ", x[i])
#   print("\n")

original:  wind speed is a reflection of the air pressure gradients <end> <start>

shuffled:  <start> the speed pressure reflection of a air wind gradients is <end>





original:  visible light makes up a fraction of all electromagnetic energy <end> <start>

shuffled:  <start> up visible light of makes fraction electromagnetic energy a all <end>





original:  most women experience a menstrual period four to six weeks after a miscarriage <end> <start>

shuffled:  <start> six to menstrual miscarriage women a most after weeks experience a four period <end>





original:  all weasels have scent glands <comma> but some are more powerful than others <end> <start>

shuffled:  <start> more powerful but some glands have weasels <comma> scent all are others than <end>





original:  some women can develop the technique of achieving ejaculation <end> <start>

shuffled:  <start> achieving can of the ejaculation women develop technique some <end>





original:  yoga improves fitness <comma> l

# Metrics

Let s be the source string and p your prediction. The quality of the results will be measured according to the following metric:

1.  look for the longest substring w between s and p
2.  compute |w|/max(|s|,|p|)

If the match is exact, the score is 1.

When computing the score, you should NOT consider the start and end tokens.



The longest common substring can be computed with the SequenceMatcher function of difflib, that allows a simple definition of our metric.

In [7]:
from difflib import SequenceMatcher

def score(s,p):
  match = SequenceMatcher(None, s, p).find_longest_match()
  return (match.size/max(len(p),len(s)))

Let's do an example.

In [ ]:
original = "at first henry wanted to be friends with the king of france"
generated = "henry wanted to be friends with king of france at the first"
print("your score is ",score(original,generated))

your score is  0.5423728813559322


The score must be computed as an average of at least 3K random examples taken form the test set.

# What to deliver

You are supposed to deliver a single notebook, suitably commented.
The notebook should describe a single model, although you may briefly discuss additional attempts you did.

The notebook should contain a full trace of the training.
Weights should be made available on request.

You must also give a clear assesment of the performance of the model, computed with the metric that has been given to you.

# Good work!

In [8]:
from keras.layers import LayerNormalization, MultiHeadAttention, Add, Conv1D, Input, Dense, Embedding
from keras.layers import GlobalAveragePooling1D, GlobalMaxPooling1D, Concatenate, LayerNormalization, Dropout
from keras.models import Model
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model

In [9]:
### Positional Encoding ###
def positional_encoding(max_sequence_len, model_dim):
    """
    :param max_sequence_len: the maximum length of the sequence
    :param model_dim: the embedding dimension
    """

    positions = np.arange(max_sequence_len)[:, np.newaxis]
    dimensions = np.arange(model_dim)[np.newaxis, :]

    angle_rates = 1 / np.power(10000, (2 * (dimensions // 2)) / np.float32(model_dim))
    angle_rads = positions * angle_rates

    # apply sin to even indices in the array; 2i
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])

    # apply cos to odd indices in the array; 2i+1
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    pos_encoding = angle_rads[np.newaxis, ...]
    return pos_encoding

In [151]:
from tensorflow.keras.layers import Input, MultiHeadAttention, Dropout, LayerNormalization, Dense, Add
from tensorflow.keras.models import Model

### Encoder Block ###
def encoder_block(latent_dim, num_heads, feed_forward_dim, dropout_rate):
    input_layer = Input(shape=(None, latent_dim))
    mask = Input(shape=(None, None), dtype='int32')  # attention mask

    # Multi-head attention
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=latent_dim // num_heads, dropout=dropout_rate)
    attention_output = attention(input_layer, input_layer, input_layer, attention_mask=mask)
    attention_output = Dropout(dropout_rate)(attention_output)
    attention_output = Add()([input_layer, attention_output])
    attention_output = LayerNormalization()(attention_output)

    # Feedforward
    outputs = Dense(feed_forward_dim, activation='relu')(attention_output)
    outputs = Dense(latent_dim)(outputs)
    outputs = Dropout(dropout_rate)(outputs)
    outputs = Add()([attention_output, outputs])
    outputs = LayerNormalization()(outputs)

    return Model([input_layer, mask], outputs) 

In [153]:
### Decoder Block ###
from tensorflow.keras.layers import Reshape
def decoder_block(latent_dim, num_heads, feed_forward_dim, dropout_rate):

    input_layer = Input(shape=(None, latent_dim)) # teacher forcing
    encoder_output = Input(shape=(None, latent_dim))
    
    # Masks
    padding_mask = Input(shape=(None, 28), dtype='int32')  
    look_ahead_mask = Input(shape=(None, 28), dtype='int32')

    # Self-attention
    attention1 = MultiHeadAttention(num_heads=num_heads, key_dim=latent_dim // num_heads, dropout=dropout_rate)
    attention_output1 = attention1(input_layer, input_layer, input_layer, attention_mask=look_ahead_mask)
    attention_output1 = Dropout(dropout_rate)(attention_output1)
    attention_output1 = Add()([input_layer, attention_output1])
    attention_output1 = LayerNormalization()(attention_output1)

    # Encoder-decoder attention
    attention2 = MultiHeadAttention(num_heads=num_heads, key_dim=latent_dim // num_heads, dropout=dropout_rate)
    attention_output2 = attention2(attention_output1, encoder_output, encoder_output, attention_mask=padding_mask)
    attention_output2 = Dropout(dropout_rate)(attention_output2)
    attention_output2 = Add()([attention_output1, attention_output2])
    attention_output2 = LayerNormalization()(attention_output2)

    # Feedforward
    outputs = Dense(feed_forward_dim, activation='relu')(attention_output2)
    outputs = Dense(latent_dim)(outputs)
    outputs = Dropout(dropout_rate)(outputs)
    outputs = Add()([attention_output2, outputs])
    outputs = LayerNormalization()(outputs)

    return Model([input_layer, encoder_output, padding_mask, look_ahead_mask], outputs)

In [154]:
### Transformer Model ###
def transformer_model(latent_dim, num_heads, feed_forward_dim, dropout_rate, max_sequence_len, num_transformer_blocks, vocab_size):
    """
    :param latent_dim: the embedding dimension
    :param num_heads: the number of heads in the multiheadattention models
    :param feed_forward_dim: the number of neurons in the feedforward network
    :param dropout_rate: the dropout rate
    :param max_sequence_len: the maximum sequence length
    :param num_transformer_blocks: the number of transformer blocks
    :param vocab_size: the size of the vocabulary
    """

    # Encoder
    encoder_inputs = Input(shape=(max_sequence_len,))
    print("Encoder input: ", encoder_inputs.shape)
    
    encoder_mask = Input(shape=(None,max_sequence_len), dtype='bool')
    print("Encoder input: ", encoder_inputs.shape)

    encoder_embedding = Embedding(vocab_size, latent_dim)(encoder_inputs)
    # pos_encoding = positional_encoding(max_sequence_len, latent_dim)
    # encoder_outputs = encoder_embedding + pos_encoding
    encoder_outputs = encoder_embedding
    print("Encoder embedding: ", encoder_embedding.shape)

    for i in range(num_transformer_blocks):
        encoder_outputs = encoder_block(latent_dim, num_heads, feed_forward_dim, dropout_rate)([encoder_outputs, encoder_mask])

    print("Encoder outputs: ", encoder_outputs.shape)

    # Decoder
    # dec_sequence_len = max_sequence_len -1
    decoder_inputs = Input(shape=(max_sequence_len,))
    print("Decoder inputs: ", decoder_inputs.shape)
    
    decoder_padding = Input(shape=(None,max_sequence_len), dtype='bool')
    look_ahead_mask = Input(shape=(None,max_sequence_len), dtype='bool')
    
    decoder_embedding = Embedding(vocab_size, latent_dim)(decoder_inputs)
    pos_encoding = positional_encoding(max_sequence_len, latent_dim)
    decoder_outputs = decoder_embedding + pos_encoding
    print("Decoder embedding: ", decoder_outputs.shape)

    for i in range(num_transformer_blocks):
        decoder_outputs = decoder_block(latent_dim, num_heads, feed_forward_dim, dropout_rate)([decoder_outputs, encoder_outputs, decoder_padding, look_ahead_mask])

    print("Decoder outputs: ", decoder_outputs.shape)

    # Attention
    # attention = MultiHeadAttention(num_heads=num_heads, key_dim=latent_dim // num_heads, dropout=dropout_rate)
    # context_vector = attention(decoder_outputs, encoder_outputs, encoder_outputs)
    # decoder_combined_context = Concatenate()([context_vector, decoder_outputs])

    # Re-adapt dimension
    decoder_outputs = Dense(latent_dim, activation='relu')(decoder_outputs)

    # Transformer block
    decoder_outputs = transformer_block(latent_dim, num_heads, feed_forward_dim, dropout_rate)(decoder_outputs)

    # Output layer
    decoder_outputs = Dense(vocab_size, activation='softmax')(decoder_outputs)

    return Model([encoder_inputs, encoder_mask, decoder_inputs, decoder_padding, look_ahead_mask], decoder_outputs)

In [15]:
def masked_loss(y_true, y_pred):
    mask = tf.math.logical_not(tf.math.equal(y_true, 0))
    loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=False, reduction='none')
    
    loss_ = loss_object(y_true, y_pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask

    return tf.reduce_sum(loss_)/tf.reduce_sum(mask)

def masked_accuracy(y_true, y_pred):
    mask = tf.math.logical_not(tf.math.equal(y_true, 0))
    correct = tf.cast(tf.math.equal(y_true, tf.argmax(y_pred, axis=-1, output_type=tf.int32)), dtype=tf.float32)
    mask = tf.cast(mask, dtype=correct.dtype)
    correct *= mask

    return tf.reduce_sum(correct)/tf.maximum(tf.reduce_sum(mask), 1)

In [16]:
class CustomSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
  def __init__(self, d_model, warmup_steps=4000):
    super().__init__()

    self.d_model = d_model
    self.d_model = tf.cast(self.d_model, tf.float32)

    self.warmup_steps = warmup_steps

  def __call__(self, step):
    step = tf.cast(step, dtype=tf.float32)
    arg1 = tf.math.rsqrt(step)
    arg2 = step * (self.warmup_steps ** -1.5)

    return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

  def get_config(self):
    return {
        'd_model': self.d_model,
        'warmup_steps': self.warmup_steps
    } 

In [155]:
latent_dim = 128
num_heads = 16
feed_forward_dim = 256
dropout_rate = 0.5
max_sequence_len = 28
num_transformer_blocks = 4
vocab_size = len(tokenizer.get_vocabulary())

model = transformer_model(latent_dim, num_heads, feed_forward_dim, dropout_rate, max_sequence_len, num_transformer_blocks, vocab_size)
model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy')
model.summary()

Encoder input:  (None, 28)
Encoder input:  (None, 28)
Encoder embedding:  (None, 28, 128)
Encoder outputs:  (None, 28, 128)
Decoder inputs:  (None, 28)
Decoder embedding:  (None, 28, 128)
Decoder outputs:  (None, 28, 128)


Model: "functional_397"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_582     │ (None, 28)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_42        │ (None, 28, 128)   │  1,280,000 │ input_layer_582[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_583     │ (None, None, 28)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_379      │ (None, 28, 128)   │    132,480 │ embedding_42[0][… │
│ (Functional)        │                   │            │ input_layer_583[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_592     │ (None, 28)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_381      │ (None, 28, 128)   │    132,480 │ functional_379[0… │
│ (Functional)        │                   │            │ input_layer_583[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_43        │ (None, 28, 128)   │  1,280,000 │ input_layer_592[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_383      │ (None, 28, 128)   │    132,480 │ functional_381[0… │
│ (Functional)        │                   │            │ input_layer_583[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_442 (Add)       │ (None, 28, 128)   │          0 │ embedding_43[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_385      │ (None, 28, 128)   │    132,480 │ functional_383[0… │
│ (Functional)        │                   │            │ input_layer_583[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_593     │ (None, None, 28)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_594     │ (None, None, 28)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_387      │ (None, 28, 128)   │    198,784 │ add_442[0][0],    │
│ (Functional)        │                   │            │ functional_385[0… │
│                     │                   │            │ input_layer_593[… │
│                     │                   │            │ input_layer_594[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_389      │ (None, 28, 128)   │    198,784 │ functional_387[0… │
│ (Functional)        │                   │            │ functional_385[0… │
│                     │                   │            │ input_layer_593[… │
│                     │                   │            │ input_layer_594[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_391      │ (None, 28, 128)   │    198,784 │ functional_389[0… │
│ (Functional)        │                   │            │ functional_385[0… │
│                     │                   │            │ input_layer_593[… │
│                     │                   │            │ input_layer_594[

 Total params: 5,324,048 (20.31 MB)

 Trainable params: 5,324,048 (20.31 MB)

 Non-trainable params: 0 (0.00 B)

In [156]:
train_generator = DataGenerator(original_data[:220000], training=True, batch_size=64)
test_generator = DataGenerator(original_data[220000:], training=False, batch_size=64)

In [ ]:
# Encoder input:  (None, 28)
# Encoder embedding:  (None, 28, latent_dim)
# Encoder outputs:  (None, 28, latent_dim)

# Decoder inputs:  (None, 28)
# Decoder embedding:  (None, 28, latent_dim)
# Decoder outputs:  (None, 28, latent_dim)

# Shape of data_batch: (batch_size, 28)
# Shape of targ_in: (batch_size, 27)
# Shape of targ_out: (batch_size, 27)

In [158]:
for layer in model.layers:
    print(f"Layer: {layer.name}")
    for variable in layer.variables:
        print(f"Variable: {variable.name}, Shape: {variable.shape}")

Layer: input_layer_582
Layer: embedding_42
Variable: embeddings, Shape: (10000, 128)
Layer: input_layer_583
Layer: functional_379
Variable: kernel, Shape: (128, 16, 8)
Variable: bias, Shape: (16, 8)
Variable: kernel, Shape: (128, 16, 8)
Variable: bias, Shape: (16, 8)
Variable: kernel, Shape: (128, 16, 8)
Variable: bias, Shape: (16, 8)
Variable: seed_generator_state, Shape: (2,)
Variable: kernel, Shape: (16, 8, 128)
Variable: bias, Shape: (128,)
Variable: seed_generator_state, Shape: (2,)
Variable: gamma, Shape: (128,)
Variable: beta, Shape: (128,)
Variable: kernel, Shape: (128, 256)
Variable: bias, Shape: (256,)
Variable: kernel, Shape: (256, 128)
Variable: bias, Shape: (128,)
Variable: seed_generator_state, Shape: (2,)
Variable: gamma, Shape: (128,)
Variable: beta, Shape: (128,)
Layer: input_layer_592
Layer: functional_381
Variable: kernel, Shape: (128, 16, 8)
Variable: bias, Shape: (16, 8)
Variable: kernel, Shape: (128, 16, 8)
Variable: bias, Shape: (16, 8)
Variable: kernel, Shape: (

In [157]:
# FIRST TRAINING - 5 EPOCHS ####
# checkpoint_path = "/content/drive/MyDrive/Colab/PRJ/transformer10_128_64_16_8_8.h5"
# checkpoint = ModelCheckpoint(checkpoint_path, save_best_only=True)
# early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)

history1 = model.fit(train_generator, epochs=1)
model.save('v2_3_128_256.keras')

(64, 28)
(64, 28)
(64, 28)
(64, 28)
(64, 28)
(64, 28)


ValueError: Exception encountered when calling Functional.call().

[1mInvalid input shape for input Tensor("data_1:0", shape=(None, 28), dtype=bool). Expected shape (None, None, 28), but input has incompatible shape (None, 28)[0m

Arguments received by Functional.call():
  • inputs=('tf.Tensor(shape=(None, 28), dtype=int32)', 'tf.Tensor(shape=(None, 28), dtype=bool)', 'tf.Tensor(shape=(None, 28), dtype=int32)', 'tf.Tensor(shape=(None, 28), dtype=bool)', 'tf.Tensor(shape=(None, 28), dtype=bool)')
  • training=True
  • mask=('None', 'None', 'None', 'None', 'None')

In [ ]:
### SECOND TRAINING - 5 EPOCHS ###
from tensorflow.keras.models import load_model

history2 = model.fit(train_generator, epochs=5)
model.save('/content/drive/MyDrive/Colab/PRJ/transformer10_128_64_16_8_8_10epochs.h5')

Epoch 1/5

3437/3437 [==============================] - 301s 87ms/step - loss: 0.0110

Epoch 2/5

3437/3437 [==============================] - 299s 87ms/step - loss: 0.0100

Epoch 3/5

3437/3437 [==============================] - 300s 87ms/step - loss: 0.0093

Epoch 4/5

3437/3437 [==============================] - 297s 86ms/step - loss: 0.0086

Epoch 5/5

3437/3437 [==============================] - 302s 88ms/step - loss: 0.0080


In [ ]:
# # predict one token at a time

# def predict(encoder_input, model, max_sequence_len):
#     decoder_input = np.zeros((1, max_sequence_len-1))
#     decoder_input[0,0] = 3

#     for i in range(1, max_sequence_len-1):
#         output = model.predict([encoder_input.reshape(1, -1), decoder_input])
#         token = np.argmax(output[0, i-1])
#         decoder_input[0, i] = token

#     return detokenizer(decoder_input)

# y_pred = predict(original_data[0], model, max_sequence_len)
# print("Predicted: ", y_pred)

In [ ]:
def clean_sentence(x):
    x = x.replace('<start>', '').replace('<end>', '').replace('<pad>', '').strip()
    return x

In [ ]:
### Predict One Token at a Time ###
def predict(encoder_input, initial_state, model, max_sequence_len=28):
    
    batch_size = encoder_input.shape[0]
    decoder_input = np.zeros((batch_size, max_sequence_len))
    bow = [[word for word in sentence if word not in [3, 2, 0]] for sentence in encoder_input]

    for i in range(batch_size):
        decoder_input[i,0] = 3 # <start> token
        # decoder_input[i,27] = 2 # <end> token

    for i in range(1, max_sequence_len):
        output = model.predict([encoder_input, decoder_input])

        # i-th word predicted, for each sentence of the batch
        # token = np.argmax(output[:, i-1, :], axis=-1)
        # token = np.argmax(output[:, 1, :], axis=-1)
        token = output[:, -1, :]
 
        # for each sentence in the batch
        for j in range(len(bow)):
            
            if len(bow[j]) == 0:
                cand_token = 2 # end token
            else:
                # choose only words that appears in the sentence
                s_pred = token[j, np.array(bow[j])]
                
                # choose the most likely word
                cand_index = np.argmax(s_pred)
                cand_token = bow[j][cand_index]
                del bow[j][cand_index]
            
            # add the predicted word to the sentence
            decoder_input[j,i] = cand_token
            
            # decoder_input[j,i] = token[j]
        # print(decoder_input[0])
    return decoder_input

In [ ]:
# (64, 28, 1000)
# (64, 28)
# (64)

# bow = (64, num words, max 28)

In [ ]:
### Show Predicted Sentences ###
x, y_true = test_generator.__getitem__(1)
y_pred = predict(x[0], x[1], model)

for true, pred in zip(y_true, y_pred):
    s_pred = clean_sentence ( detokenizer([pred])[0] )
    s_true = clean_sentence ( detokenizer([true])[0] )
    
    print("Predicted: ", s_pred)
    print("True: ", s_true)
    print('\n')

In [14]:
# from tensorflow.keras.models import load_model
# model = load_model('/content/drive/MyDrive/Colab/PRJ/transformer10_128_64_16_8_8_10epochs.h5')

In [ ]:
import random
def evaluate_baseline(test_generator, score_func):
    scores = []

    for i in range(len(test_generator)):
        x, y_true = test_generator.__getitem__(i)
        shuffled = x[0].copy()
        random.shuffle(shuffled)

        for true, shuffle in zip(detokenizer(y_true), detokenizer(shuffled)):
            scores.append(score_func(true, shuffle))
    return np.mean(scores), np.std(scores)

baseline_mean, baseline_std = evaluate_baseline(test_generator, score)
print (f'Baseline score: {baseline_mean}')
print (f'Baseline Stdev: {baseline_std}')
print (f'Baseline 3*Stdev: {baseline_std * 3}')

Baseline score: 0.09287600404534446

Baseline Stdev: 0.028945709804022023

Baseline 3*Stdev: 0.08683712941206606


In [ ]:
def evaluate_model(model, test_generator, score_func):
    scores = []

    for i in range(len(test_generator)):
        print("Test :", i)
        x, y_true = test_generator.__getitem__(i)
        y_pred = predict(x[0], x[1], model)
        
        for true, pred in zip(y_true, y_pred):
            s_pred = clean_sentence ( detokenizer([pred])[0] )
            s_true = clean_sentence ( detokenizer([true])[0] )
            
            scores.append(score_func(s_true, s_pred))
    return np.mean(scores), np.std(scores)

average_score, stdev_score = evaluate_model(model, test_generator, score)
print(f'Average score: {average_score}')
print(f'Stdev: {stdev_score}')
print(f'3*Stdev: {stdev_score * 3}')

In [ ]:
# compute the improvement over random guess
improvement = (average_score - baseline_mean) / baseline_mean
print(f'Improvement over random guess: {improvement}')

Improvement over random guess: 8.618192898194717
